# 1. Analysis - Relative value

This notebook compares the current Single Asset Profile ticker against a GICS-defined peer set using trailing valuation multiples and yield metrics from Financial Modeling Prep. Peer selection follows the same S&P 1500 / GICS company metadata process used in `Equities.ipynb`; FMP is used afterward only to collect valuation data for the selected symbols.

Run the notebook from top to bottom after updating the local `TICKER` parameter near the top of this notebook. Configure `peer_group_level` as `Sub-Industry`, `Industry`, `Industry Group`, or `Sector` to control how broad the peer set should be. For ETFs or securities outside the S&P 1500 GICS universe, use `manual_peer_symbols` or switch to a company ticker.


In [ ]:
# 2. Setup
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

peer_analysis_params = {
    "ticker_str": "AAPL",
    "peer_limit": 12,
    "peer_group_level": "Sub-Industry",
    "minimum_peer_count": 4,
    "peer_capitalizations": None,  # None = include Large / Mid / Small Cap; use "same_as_target" for one cap bucket
}

TICKER = peer_analysis_params["ticker_str"]
PEER_LIMIT = peer_analysis_params["peer_limit"]
PEER_GROUP_LEVEL = peer_analysis_params["peer_group_level"]
MINIMUM_PEER_COUNT = peer_analysis_params["minimum_peer_count"]
PEER_CAPITALIZATIONS = peer_analysis_params["peer_capitalizations"]

# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()

from Quantapp.data import (
    GICSDataClient,
    build_capitalization_count_table,
    build_gics_peer_frames,
    build_gics_peer_table,
    choose_gics_peer_level,
    get_peer_analysis_data,
    normalize_peer_symbol,
    select_gics_peer_rows,
)


TICKER

## 3. Build the Peer Universe

This cell builds the peer hierarchy from the same S&P 1500 / GICS company table used by `Equities.ipynb`. It first saves and prints all peers in the selected ticker's Sector across Large, Mid, and Small Cap, then narrows that DataFrame into Industry Group, Industry, and Sub-Industry tables. FMP is queried only after those local GICS tables are visible, using the configured GICS level and peer limit for valuation metrics.


In [ ]:
# 4. Build GICS peer hierarchy, then fetch valuation metrics
manual_peer_symbols = []
gics_data = GICSDataClient(save_path=PROJECT_ROOT)

def display_peer_level(level, peer_table, target_row):
    print(f"{level}: {target_row[level]} ({len(peer_table)} peers)")
    display(peer_table)

all_companies = gics_data.retrieve_companies()
gics_peer_context = build_gics_peer_frames(
    TICKER,
    companies=all_companies,
    capitalizations=PEER_CAPITALIZATIONS,
    symbol_label='FMP Symbol',
)
target_gics_row = gics_peer_context.target_row
peer_capitalization_filter = gics_peer_context.capitalization_filter
peer_summary = gics_peer_context.summary

sector_peer_universe = gics_peer_context.frame('Sector')
industry_group_peer_universe = gics_peer_context.frame('Industry Group')
industry_peer_universe = gics_peer_context.frame('Industry')
sub_industry_peer_universe = gics_peer_context.frame('Sub-Industry')

sector_peer_table = gics_peer_context.table('Sector')
industry_group_peer_table = gics_peer_context.table('Industry Group')
industry_peer_table = gics_peer_context.table('Industry')
sub_industry_peer_table = gics_peer_context.table('Sub-Industry')

print(f"GICS peer hierarchy for {normalize_peer_symbol(TICKER)} before FMP lookup")
print(f"Capitalization filter: {peer_capitalization_filter}")
print("Peer count summary by filter step:")
display(peer_summary)
display_peer_level('Sector', sector_peer_table, target_gics_row)
display_peer_level('Industry Group', industry_group_peer_table, target_gics_row)
display_peer_level('Industry', industry_peer_table, target_gics_row)
display_peer_level('Sub-Industry', sub_industry_peer_table, target_gics_row)

peer_group_level_used, selected_gics_peer_universe = choose_gics_peer_level(
    gics_peer_context,
    preferred_level=PEER_GROUP_LEVEL,
    minimum_peer_count=MINIMUM_PEER_COUNT,
)
peer_group_value = target_gics_row[peer_group_level_used]
gics_peer_universe = select_gics_peer_rows(
    selected_gics_peer_universe,
    peer_count=max(PEER_LIMIT - 1, 0),
    target_capitalization=target_gics_row.get('Capitalization'),
)
gics_peer_symbols = gics_peer_universe['Normalized Symbol'].tolist()
peer_source_label = (
    f"GICS {peer_group_level_used}: {peer_group_value}; Capitalization: {peer_capitalization_filter} "
    f"({len(selected_gics_peer_universe)} candidates before peer_limit)"
)

selected_peer_table = build_gics_peer_table(gics_peer_universe, symbol_label='FMP Symbol')
selected_peer_capitalization_counts = build_capitalization_count_table(selected_peer_table)
print(f"Selected {peer_group_level_used} peers sent to FMP valuation lookup:")
display(selected_peer_table)
display(selected_peer_capitalization_counts)

peer_analysis = get_peer_analysis_data(
    TICKER,
    peer_limit=PEER_LIMIT,
    peer_universe_symbols=gics_peer_symbols,
    peer_source_label=peer_source_label,
    manual_peer_symbols=manual_peer_symbols,
)

SYMBOL = peer_analysis.symbol
company_name = peer_analysis.company_name
industry_name = target_gics_row.get('Industry', peer_analysis.industry_name)
sector_name = target_gics_row.get('Sector', peer_analysis.sector_name)
target_exchange = peer_analysis.target_exchange
target_market_cap = peer_analysis.target_market_cap
quote_snapshot = peer_analysis.quote
profile_snapshot = peer_analysis.profile
raw_peer_symbols = peer_analysis.raw_peer_symbols
filtered_peer_symbols = peer_analysis.filtered_peer_symbols
peer_symbols = peer_analysis.peer_symbols
peer_filter_summary = gics_peer_universe
peer_overview = peer_analysis.peer_overview.copy()
peer_overview['gicsPeerLevel'] = peer_group_level_used
peer_overview['gicsPeerGroup'] = peer_group_value
peer_overview['gicsCapitalizationFilter'] = peer_capitalization_filter
peer_overview['gicsSectorPeerCount'] = len(sector_peer_universe)
peer_overview['gicsIndustryGroupPeerCount'] = len(industry_group_peer_universe)
peer_overview['gicsIndustryPeerCount'] = len(industry_peer_universe)
peer_overview['gicsSubIndustryPeerCount'] = len(sub_industry_peer_universe)
peer_overview['gicsCandidateCount'] = len(selected_gics_peer_universe)
peer_overview['gicsFmpPeerCount'] = len(gics_peer_universe)
peer_metrics = peer_analysis.peer_metrics
relative_value_summary = peer_analysis.relative_value_summary
relative_value = peer_analysis.relative_value

peer_overview


In [ ]:
# 5. Review peer valuation metrics
peer_metrics

In [ ]:
# 6. Summarize relative value vs peer medians
relative_value_summary

In [ ]:
# 7. Plot relative valuation vs peer medians
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_relative_value_vs_peer_medians

relative_value_plot = plot_relative_value_vs_peer_medians(relative_value, company_name=company_name)
relative_value_plot.show(config={"responsive": True, "displaylogo": False})